# 模型质量抽样检查

下载少量模型 → 背景检测 → 可视化 → HTML 报告

In [ ]:
# 0. 安装依赖
!pip install objaverse trimesh matplotlib --quiet

In [ ]:
# 1. 加载 annotations，筛选 30 个椅子模型
import objaverse
import os
os.makedirs('/content/test_models', exist_ok=True)
os.makedirs('/content/inspection_output', exist_ok=True)

print('Loading annotations...')
annotations = objaverse.load_annotations()

# 筛选：chair 类别 + 有较长描述
chair_uids = []
for uid, anno in annotations.items():
    name = str(anno.get('name', '')).lower()
    desc = str(anno.get('description', ''))
    tags = ' '.join([str(t.get('name', '')).lower() for t in anno.get('tags', [])])
    text = name + ' ' + tags
    if 'chair' in text and len(desc) > 50:
        chair_uids.append(uid)
    if len(chair_uids) >= 30:
        break

print(f'Found {len(chair_uids)} chair models with rich descriptions')

# 只下载前 10 个做检查
sample_uids = chair_uids[:10]
print(f'Sampling {len(sample_uids)} models for inspection')

In [ ]:
# 2. 下载这 10 个模型
print('Downloading models...')
downloaded = objaverse.load_objects(
    uids=sample_uids,
    download_processes=4
)

import shutil
model_dir = '/content/test_models'
for uid, path in downloaded.items():
    if os.path.exists(path):
        shutil.copy(path, os.path.join(model_dir, f'{uid}.glb'))

actual_files = os.listdir(model_dir)
print(f'Downloaded {len(actual_files)} models to {model_dir}')
print(f'Sample files: {actual_files[:3]}')

In [ ]:
# 3. 复制检查工具代码
# (把 voxer/inspect.py 的内容粘贴到这里，避免上传整个包)
%%writefile /content/inspect_tool.py
"""Model inspection tool - standalone version for Colab"""

import os, json, math, warnings, gc
import numpy as np
from typing import List, Dict, Optional, Tuple
from dataclasses import dataclass, field

warnings.filterwarnings("ignore", category=UserWarning)


@dataclass
class MeshInfo:
    name: str
    index: int
    num_vertices: int
    num_faces: int
    centroid: np.ndarray
    bbox_min: np.ndarray
    bbox_max: np.ndarray
    volume: float
    extent: float
    flatness: float
    is_ground_plane: bool = False
    is_background: bool = False
    flag_reasons: List[str] = field(default_factory=list)


def load_glb_parts(file_path):
    import trimesh
    try:
        scene = trimesh.load(file_path, force="scene")
    except Exception:
        try:
            scene = trimesh.load(file_path, force="mesh")
            if scene is None:
                return None
            return [scene], scene
        except Exception:
            return None

    if not isinstance(scene, trimesh.Scene):
        if scene is not None:
            return [scene], scene
        return None

    geometry = list(scene.geometry.values())
    if len(geometry) == 0:
        return None

    combined = trimesh.util.concatenate(geometry)
    return geometry, combined


def analyze_mesh(mesh, idx):
    vertices = mesh.vertices
    name = mesh.metadata.get("name", f"mesh_{idx}") if hasattr(mesh, "metadata") else f"mesh_{idx}"
    bbox_min = vertices.min(axis=0)
    bbox_max = vertices.max(axis=0)
    centroid = vertices.mean(axis=0)
    extent = np.linalg.norm(bbox_max - bbox_min)
    sizes = bbox_max - bbox_min
    volume = sizes[0] * sizes[1] * sizes[2]
    sorted_sizes = sorted(sizes)
    flatness = sorted_sizes[2] / (sorted_sizes[0] + 1e-8)

    return MeshInfo(
        name=name, index=idx,
        num_vertices=len(vertices),
        num_faces=len(mesh.faces) if mesh.faces is not None else 0,
        centroid=centroid, bbox_min=bbox_min, bbox_max=bbox_max,
        volume=volume, extent=extent, flatness=flatness,
    )


def detect_background_parts(meshes, plane_flatness_threshold=50.0,
                             volume_ratio_threshold=0.7, centroid_distance_std=2.5):
    if len(meshes) <= 1:
        return meshes

    volumes = [m.volume for m in meshes]
    centroids = np.array([m.centroid for m in meshes])
    median_centroid = np.median(centroids, axis=0)
    centroid_distances = np.linalg.norm(centroids - median_centroid, axis=1)
    mean_dist = np.mean(centroid_distances)
    std_dist = np.std(centroid_distances) if len(centroid_distances) > 0 else 1

    total_vol = max(volumes)
    for i, m in enumerate(meshes):
        reasons = []
        if m.flatness > plane_flatness_threshold:
            m.is_ground_plane = True
            reasons.append(f"flatness={m.flatness:.0f}")
        if volumes[i] > total_vol * volume_ratio_threshold and len(meshes) > 1:
            reasons.append(f"volume {volumes[i]/total_vol:.0%}")
        if centroid_distances[i] > mean_dist + centroid_distance_std * std_dist and len(meshes) > 1:
            reasons.append(f"offset {centroid_distances[i]:.2f}")
        if reasons:
            m.is_background = True
            m.flag_reasons = reasons

    return meshes


def render_mesh_views(mesh, title="", color="steelblue", sample_points=20000):
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt

    try:
        points, _ = mesh.sample(sample_points, return_index=True)
    except Exception:
        verts = mesh.vertices
        if len(verts) > sample_points:
            idx = np.random.choice(len(verts), sample_points, replace=False)
            points = verts[idx]
        else:
            points = verts

    fig, axes = plt.subplots(1, 3, figsize=(10, 3.5))
    views = [("Top (XY)", 0, 1), ("Front (XZ)", 0, 2), ("Side (YZ)", 1, 2)]
    for ax, (label, d1, d2) in zip(axes, views):
        ax.scatter(points[:, d1], points[:, d2], s=0.5, alpha=0.5,
                   c=color, edgecolors="none")
        ax.set_title(label, fontsize=10)
        ax.set_aspect("equal")
        ax.axis("equal")

    if title:
        fig.suptitle(title, fontsize=11, fontweight="bold")
    plt.tight_layout()
    return fig


def render_part_overview(meshes, infos, title="", sample_points=5000):
    import matplotlib.pyplot as plt
    colors = plt.cm.tab10(np.linspace(0, 1, max(len(meshes), 1)))
    fig, axes = plt.subplots(1, 3, figsize=(12, 4))
    views = [("Top", 0, 1), ("Front", 0, 2), ("Side", 1, 2)]

    for ax, (label, d1, d2) in zip(axes, views):
        for i, (mesh, info) in enumerate(zip(meshes, infos)):
            try:
                pts, _ = mesh.sample(sample_points, return_index=True)
            except Exception:
                verts = mesh.vertices
                if len(verts) > sample_points:
                    idx = np.random.choice(len(verts), sample_points, replace=False)
                    pts = verts[idx]
                else:
                    pts = verts

            lbl = f"{info.name[:20]}"
            if info.is_background:
                lbl += " [!]"
            ax.scatter(pts[:, d1], pts[:, d2], s=0.3, alpha=0.4,
                       c=[colors[i]], edgecolors="none", label=lbl)
        ax.set_title(label, fontsize=10)
        ax.set_aspect("equal")

    axes[0].legend(loc="upper right", fontsize=5, ncol=2, bbox_to_anchor=(1.0, -0.05))
    if title:
        fig.suptitle(title, fontsize=11, fontweight="bold")
    return fig


def inspect_single_model(file_path, output_dir, sample_points=20000):
    import trimesh
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt

    os.makedirs(output_dir, exist_ok=True)
    uid = os.path.splitext(os.path.basename(file_path))[0]
    result = {"uid": uid, "num_parts": 0, "flagged": 0, "parts": [], "images": []}

    parts = load_glb_parts(file_path)
    if parts is None:
        result["error"] = "failed to load"
        return result

    meshes, combined = parts
    result["num_parts"] = len(meshes)
    infos = [analyze_mesh(m, i) for i, m in enumerate(meshes)]
    infos = detect_background_parts(infos)

    for info in infos:
        result["parts"].append({
            "name": info.name, "vertices": info.num_vertices,
            "volume": float(info.volume), "flatness": float(info.flatness),
            "is_bg": info.is_background, "reasons": info.flag_reasons
        })
        if info.is_background:
            result["flagged"] += 1

    if len(meshes) > 1:
        fig = render_part_overview(meshes, infos,
            title=f"{uid[:30]} | {len(meshes)} parts | {result['flagged']} flagged")
        path = os.path.join(output_dir, f"{uid}_parts.png")
        fig.savefig(path, dpi=100, bbox_inches="tight")
        plt.close(fig)
        result["images"].append(path)

    fig = render_mesh_views(combined,
        title=f"{uid[:30]} | Full model | {len(meshes)} sub-meshes")
    path = os.path.join(output_dir, f"{uid}_full.png")
    fig.savefig(path, dpi=100, bbox_inches="tight")
    plt.close(fig)
    result["images"].append(path)

    return result


In [ ]:
# 4. 批量检查所有模型
from inspect_tool import inspect_single_model
import glob

model_dir = '/content/test_models'
output_dir = '/content/inspection_output'

files = sorted(glob.glob(os.path.join(model_dir, '*.glb')))
print(f'Inspecting {len(files)} models...\n')

all_results = []
for f in files:
    result = inspect_single_model(f, output_dir)
    all_results.append(result)

    status = "OK" if result["flagged"] == 0 else f"FLAG ({result['flagged']} parts)"
    detail = ""
    for p in result.get("parts", []):
        if p["is_bg"]:
            detail += f"  [{p['name'][:25]}] {', '.join(p['reasons'])}\n"
    if detail:
        print(f"{result['uid'][:40]:<42} {status}")
        print(detail, end="")
    else:
        print(f"{result['uid'][:40]:<42} {status} ({result['num_parts']} parts)")

# 统计
clean = sum(1 for r in all_results if r["flagged"] == 0)
flagged = sum(1 for r in all_results if r["flagged"] > 0)
failed = sum(1 for r in all_results if "error" in r)
print(f"\nResult: {clean} clean, {flagged} flagged, {failed} failed")

In [ ]:
# 5. 展示一个有问题的模型 vs 一个没问题的模型
from IPython.display import Image, display
import glob

output_dir = '/content/inspection_output'

# 找一个被标记的模型
flagged_results = [r for r in all_results if r.get("flagged", 0) > 0]
clean_results = [r for r in all_results if r.get("flagged", 0) == 0 and "error" not in r]

if flagged_results:
    r = flagged_results[0]
    print(f"\nFlagged model: {r['uid'][:50]}")
    for p in r["parts"]:
        tag = "[BACKGROUND]" if p["is_bg"] else "[OK]"
        print(f"  {tag} {p['name'][:30]:<32} verts={p['vertices']:>6} vol={p['volume']:.2e} flat={p['flatness']:.0f}")

    # 显示对比图
    parts_img = glob.glob(os.path.join(output_dir, f"{r['uid']}_parts.png"))
    full_img = glob.glob(os.path.join(output_dir, f"{r['uid']}_full.png"))
    if parts_img:
        print("\nParts overview (不同颜色=不同子mesh, [!]标记=疑似背景):")
        display(Image(parts_img[0]))
    if full_img:
        print("Full model (combined):")
        display(Image(full_img[0]))

if clean_results:
    r = clean_results[0]
    print(f"\nClean model: {r['uid'][:50]} ({r['num_parts']} parts, all OK)")
    for p in r["parts"]:
        print(f"  [OK] {p['name'][:30]:<32} verts={p['vertices']:>6}")
    full_img = glob.glob(os.path.join(output_dir, f"{r['uid']}_full.png"))
    if full_img:
        display(Image(full_img[0]))

if not flagged_results and not clean_results:
    print("No successful inspections. Check errors above.")